In [99]:
# ============================================================
# 1️⃣ Imports and Load CSV files
# ============================================================

import pandas as pd
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
import pickle

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.utils import resample

In [101]:
# Folder where CSV files are located
folder_path = r"C:\Users\arsha\Downloads\archive (3)"
all_files = sorted(glob.glob(os.path.join(folder_path, "*.csv")))

# Load all CSVs and rename service names as Microservice A, B, C...
df_list = []
for i, file in enumerate(all_files):
    temp = pd.read_csv(file)
    temp['microservice'] = f"Microservice {chr(65 + i)}"  # Microservice A, B, C...
    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)
print("Data shape:", df.shape)
df.head(10)


Data shape: (2280, 3)


,datetime,cpu,microservice
0,2017-01-27 18:42:00,1.14,Microservice A
1,2017-01-27 18:43:00,1.10,Microservice A
2,2017-01-27 18:44:00,1.09,Microservice A
3,2017-01-27 18:45:00,1.08,Microservice A
4,2017-01-27 18:46:00,1.08,Microservice A
5,2017-01-27 18:47:00,1.08,Microservice A
6,2017-01-27 18:48:00,1.15,Microservice A
7,2017-01-27 18:49:00,1.13,Microservice A
8,2017-01-27 18:50:00,1.09,Microservice A
9,2017-01-27 18:51:00,1.06,Microservice A


In [103]:
# ============================================================
# 2️⃣ Preprocessing: datetime, sorting, hours_since_deploy
# ============================================================

# Convert datetime column to pandas datetime
df['datetime'] = pd.to_datetime(df['datetime'])

# Sort by microservice and datetime
df = df.sort_values(by=['microservice', 'datetime']).reset_index(drop=True)

# Calculate hours since deployment per microservice
df['hours_since_deploy'] = df.groupby('microservice')['datetime'] \
    .transform(lambda x: (x - x.min()).dt.total_seconds() / 3600)

# Quick check
df.head()


,datetime,cpu,microservice,hours_since_deploy
0,2017-01-27 18:42:00,1.14,Microservice A,0.000000
1,2017-01-27 18:43:00,1.10,Microservice A,0.016667
2,2017-01-27 18:44:00,1.09,Microservice A,0.033333
3,2017-01-27 18:45:00,1.08,Microservice A,0.050000
4,2017-01-27 18:46:00,1.08,Microservice A,0.066667


In [104]:
# ============================================================
# 3️⃣ Simulate post-deployment CPU instability & rolling stats
# ============================================================

# Set seed for reproducibility
np.random.seed(42)

# Simulate CPU instability after 12 hours
stress_mask = df['hours_since_deploy'] > 12
df.loc[stress_mask, 'cpu'] += np.random.normal(
    loc=0.4,     # average CPU increase
    scale=0.25,  # instability
    size=stress_mask.sum()
)

# Ensure CPU values are non-negative
df['cpu'] = df['cpu'].clip(lower=0)

# Compute CPU difference from previous row
df['cpu_diff'] = df.groupby('microservice')['cpu'].diff().fillna(0)

# Rolling mean and std (window=5)
df['cpu_rolling_mean'] = df.groupby('microservice')['cpu'].rolling(window=5).mean().reset_index(0, drop=True)
df['cpu_rolling_std'] = df.groupby('microservice')['cpu'].rolling(window=5).std().reset_index(0, drop=True)

# Fill NaN values from rolling operations
df[['cpu_rolling_mean', 'cpu_rolling_std']] = df[['cpu_rolling_mean', 'cpu_rolling_std']].fillna(0)

# Quick check
df.head()


,datetime,cpu,microservice,hours_since_deploy,cpu_diff,cpu_rolling_mean,cpu_rolling_std
0,2017-01-27 18:42:00,1.14,Microservice A,0.000000,0.00,0.000,0.0000
1,2017-01-27 18:43:00,1.10,Microservice A,0.016667,-0.04,0.000,0.0000
2,2017-01-27 18:44:00,1.09,Microservice A,0.033333,-0.01,0.000,0.0000
3,2017-01-27 18:45:00,1.08,Microservice A,0.050000,-0.01,0.000,0.0000
4,2017-01-27 18:46:00,1.08,Microservice A,0.066667,0.00,1.098,0.0249


In [105]:
# ============================================================
# 4️⃣ Define risk_flag and balance classes
# ============================================================

# Thresholds for risk (70th percentile for better positive samples)
CPU_THRESHOLD = df['cpu'].quantile(0.70)
STD_THRESHOLD = df['cpu_rolling_std'].quantile(0.70)

# Risk flag: 1 = risky, 0 = normal (OR logic for more positives)
df['risk_flag'] = ((df['cpu'] > CPU_THRESHOLD) | (df['cpu_rolling_std'] > STD_THRESHOLD)).astype(int)

# Check distribution
print("Class distribution:\n", df['risk_flag'].value_counts())

# Upsample minority class to balance
df_majority = df[df['risk_flag'] == 0]
df_minority = df[df['risk_flag'] == 1]

df_minority_upsampled = resample(
    df_minority,
    replace=True,
    n_samples=len(df_majority),
    random_state=42
)

df_balanced = pd.concat([df_majority, df_minority_upsampled])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# Check balanced class counts
print("Balanced class distribution:\n", df_balanced['risk_flag'].value_counts())


Class distribution:
 risk_flag
0    1434
1     846
Name: count, dtype: int64
Balanced class distribution:
 risk_flag
0    1434
1    1434
Name: count, dtype: int64


In [106]:
# ============================================================
# 5️⃣ Feature Selection & Train-Test Split
# ============================================================

# Features for ML
features = ['cpu', 'cpu_diff', 'cpu_rolling_mean', 'cpu_rolling_std', 'hours_since_deploy']
X = df_balanced[features]
y = df_balanced['risk_flag']

# Train-test split (time series aware)
# shuffle=False ensures the model sees the correct temporal order
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, shuffle=False
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print("Training class distribution:\n", y_train.value_counts())
print("Test class distribution:\n", y_test.value_counts())



Training set shape: (2151, 5)
Test set shape: (717, 5)
Training class distribution:
 risk_flag
0    1082
1    1069
Name: count, dtype: int64
Test class distribution:
 risk_flag
1    365
0    352
Name: count, dtype: int64


In [107]:
# ============================================================
# 6️⃣ Model Training & Evaluation
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import f1_score

# Define models
models = {
    "Logistic Regression": Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000))
    ]),
    "Decision Tree": Pipeline([
        ('model', DecisionTreeClassifier(random_state=42))
    ]),
    "Random Forest": Pipeline([
        ('model', RandomForestClassifier(random_state=42))
    ]),
    "Gradient Boosting": Pipeline([
        ('model', GradientBoostingClassifier(random_state=42))
    ])
}

# Train & evaluate
results = {}
for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    results[name] = f1_score(y_test, y_pred)

# Display F1 scores
results_df = pd.DataFrame(list(results.items()), columns=['Model', 'F1 Score']).sort_values(by='F1 Score', ascending=False)
print("Model F1 Scores:\n", results_df)


Model F1 Scores:
                  Model  F1 Score
1        Decision Tree  1.000000
2        Random Forest  1.000000
3    Gradient Boosting  1.000000
0  Logistic Regression  0.972527


In [108]:
# ============================================================
# 7️⃣ Logistic Regression Hyperparameter Tuning
# ============================================================

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix

# Pipeline with scaling + logistic regression
log_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

# Hyperparameter grid
param_grid = {
    'model__C': [0.01, 0.1, 1, 10],
    'model__penalty': ['l2'],
    'model__solver': ['lbfgs']
}

# Grid search with 5-fold CV
grid = GridSearchCV(log_pipe, param_grid, cv=5, scoring='f1', n_jobs=-1)
grid.fit(X_train, y_train)

# Best model
best_model = grid.best_estimator_

# Predict on test set
y_pred = best_model.predict(X_test)

# Classification metrics
print("\nClassification Report (Final Logistic Regression):")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Predict risk probability for all data

df['risk_probability'] = best_model.predict_proba(df[features])[:, 1]



Classification Report (Final Logistic Regression):
              precision    recall  f1-score   support

           0       0.97      0.97      0.97       352
           1       0.98      0.97      0.97       365

    accuracy                           0.97       717
   macro avg       0.97      0.97      0.97       717
weighted avg       0.97      0.97      0.97       717

Confusion Matrix:
[[343   9]
 [ 11 354]]


C:\Users\arsha\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [109]:
# ============================================================
# 8️⃣ Normalize Risk, Assign Risk Levels, Estimate Hours to Failure
# ============================================================

# Initialize normalized risk score
df['risk_score'] = 0.0

# Normalize risk_probability per microservice (0–1 scale)
for service in df['microservice'].unique():
    mask = df['microservice'] == service
    min_p = df.loc[mask, 'risk_probability'].min()
    max_p = df.loc[mask, 'risk_probability'].max()
    
    if max_p > min_p:
        df.loc[mask, 'risk_score'] = (df.loc[mask, 'risk_probability'] - min_p) / (max_p - min_p)
    else:
        df.loc[mask, 'risk_score'] = 0.0

# Risk score as percentage (for KPIs)
df['risk_score_percent'] = df['risk_score'] * 100

# Assign risk levels based on normalized score
def risk_level(score):
    if score < 0.3:
        return "Safe"
    elif score < 0.6:
        return "Warning"
    else:
        return "High Risk"

df['risk_level'] = df['risk_score'].apply(risk_level)

# Estimate hours to failure (first time crossing high-risk threshold)
df['estimated_hours_to_failure'] = np.nan

for service in df['microservice'].unique():
    service_df = df[df['microservice'] == service]
    high_risk_times = service_df[service_df['risk_score'] > 0.6]['hours_since_deploy']
    
    if not high_risk_times.empty:
        first_risk_time = high_risk_times.min()
        df.loc[df['microservice'] == service, 'estimated_hours_to_failure'] = first_risk_time - df['hours_since_deploy']

# Clip negative values
df['estimated_hours_to_failure'] = df['estimated_hours_to_failure'].clip(lower=0)

# Check final DataFrame
df.head()


,datetime,cpu,microservice,hours_since_deploy,cpu_diff,cpu_rolling_mean,cpu_rolling_std,risk_flag,risk_probability,risk_score,risk_score_percent,risk_level,estimated_hours_to_failure
0,2017-01-27 18:42:00,1.14,Microservice A,0.000000,0.00,0.000,0.0000,0,0.001155,0.001155,0.115468,Safe,1.400000
1,2017-01-27 18:43:00,1.10,Microservice A,0.016667,-0.04,0.000,0.0000,0,0.000600,0.000600,0.059993,Safe,1.383333
2,2017-01-27 18:44:00,1.09,Microservice A,0.033333,-0.01,0.000,0.0000,0,0.000549,0.000549,0.054856,Safe,1.366667
3,2017-01-27 18:45:00,1.08,Microservice A,0.050000,-0.01,0.000,0.0000,0,0.000475,0.000474,0.047433,Safe,1.350000
4,2017-01-27 18:46:00,1.08,Microservice A,0.066667,0.00,1.098,0.0249,0,0.000570,0.000570,0.057014,Safe,1.333333


In [110]:
# ============================================================
# 9️⃣ Save Final Model and Dataset for Dashboard
# ============================================================

import pickle

# Save final Logistic Regression model
with open("post_deploy_risk_model_final.pkl", "wb") as f:
    pickle.dump(best_model, f)

# Save dataset with risk scores and levels for dashboard (Streamlit / Tableau)
df.to_csv("post_deploy_predictions_final.csv", index=False)

print("✅ Model and dataset saved successfully")



✅ Model and dataset saved successfully
